[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/blue/notebooks/blue_pharmit_hits.ipynb)

# From Pharmit hits to a list of compounds to order

**Blue group · Cryptosporidiosis**

The Pharmit search returns tens of thousands of matches, but most of them are the same few molecules
over and over. This notebook turns that pile into a short list where every entry is a genuinely
different compound you could buy.

## What you will do

- Load the search results and read every matching shape
- Collapse the repeats so each catalogue compound appears once
- Merge molecules that are the same thing written differently
- Drop near-copies, keeping the best match of each family
- Save the list, ready for the models in the next notebook

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "blue"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Load the search results

Pharmit returns its hits as **SDF** files: long text files holding one record per match, each with
3D coordinates. The two files from the CpABC1 search are 28 MB and 140 MB, far too large for the
repository, so they live in the group's Drive folder under
**Projects/BlueTeam/Data/provisional_pharmit_results**.

Download them from there first. In Colab, run the cell and upload both when asked. Running locally,
put them in `data/downloads/` instead and the cell will find them.

> **Note:** the upload takes a few minutes, and Colab shows no progress bar until it finishes.

In [ ]:
import sys
from pathlib import Path

DOWNLOADS = Path("data/downloads")
DOWNLOADS.mkdir(parents=True, exist_ok=True)
PATTERN = "provisional_pharmit_query_results_*.sdf"

if "google.colab" in sys.modules and not list(DOWNLOADS.glob(PATTERN)):
    from google.colab import files
    print("Upload the two provisional_pharmit_query_results_*.sdf files from Drive")
    for name in files.upload():
        Path(name).rename(DOWNLOADS / name)

SDF_FILES = sorted(DOWNLOADS.glob(PATTERN))
for path in SDF_FILES:
    print(f"{path.name}: {path.stat().st_size / 1e6:.0f} MB")

## 2. Read every match

Each record in the file is one **pose**: one shape of one compound, placed in the pocket. A single
compound usually appears many times, because there are many ways to fold it and several ways to
build it.

Two parts of each record matter:

- The **title line** lists every catalogue the compound is sold in, for example
  `MolPort-016-672-966 12133290 MCULE-1421397642 ZINC000012133290`. We keep the **MolPort** numbers,
  which are what you order with.
- The **`rmsd`** value says how well the pose fits the pharmacophore, in ångströms. Lower is better.

Here is the first title line, before we read the rest.

In [ ]:
with open(SDF_FILES[0]) as fh:
    print(fh.readline().strip())

Now read every record. For each one we work out the molecule's **canonical SMILES**: a way of
writing a molecule as a line of text where the same molecule always comes out identical, which is
what lets us spot repeats. We also pull out the MolPort numbers with a text pattern.

This takes a minute or two.

In [ ]:
import re
import pandas as pd
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")
ID_RE = re.compile(r"(?i)molport-(\d{3}-\d{3}-\d{3})")

records = []
for path in SDF_FILES:
    with open(path, "rb") as fh:
        for mol in Chem.ForwardSDMolSupplier(fh):
            Chem.AssignStereochemistryFrom3D(mol)
            ids = sorted(f"MolPort-{m}" for m in ID_RE.findall(mol.GetProp("_Name")))
            records.append((Chem.MolToSmiles(mol), ids, float(mol.GetProp("rmsd"))))

poses = pd.DataFrame(records, columns=["smiles", "molport_ids", "rmsd"])
print(f"{len(poses)} poses, {(poses.molport_ids.str.len() == 0).sum()} without a MolPort number")
poses.head()

## 3. One row per molecule

Different shapes of the same molecule now share a SMILES, so grouping by it collapses them. For
each molecule we keep every MolPort number it was sold under, and its best (lowest) `rmsd`, which
section 6 uses to decide which molecules to keep.

In [ ]:
structures = poses.groupby("smiles").agg(
    molport_ids=("molport_ids", lambda col: sorted(set().union(*col))),
    rmsd=("rmsd", "min"),
).reset_index()
structures["molport_id"] = structures.molport_ids.str[0]

print(f"{len(structures)} molecules from {len(poses)} poses")
print(f"{(structures.molport_ids.str.len() > 1).sum()} are sold under more than one number")

## 4. One row per catalogue compound

Some MolPort numbers still appear on more than one row. Many molecules come in **mirror-image
forms**, like a left and a right hand: same parts, arranged the other way round. When a supplier
does not say which form it sells, Pharmit builds both and tests each, so one product becomes
several rows.

In [ ]:
per_id = structures.molport_id.value_counts()
print(f"{(per_id > 1).sum()} MolPort numbers have more than one form")
structures[structures.molport_id == per_id[per_id == 2].index[0]][["smiles", "molport_id", "rmsd"]]

The `@` and `@@` marks in those two lines of text are what tells the forms apart. Ordering the
product gets you whatever the supplier happens to have, so we keep one form per number, chosen at
random. The seed is fixed, so the notebook makes the same choice every time it runs.

In [ ]:
hits = (structures.sort_values(["molport_id", "smiles"])
        .sample(frac=1, random_state=42)
        .drop_duplicates("molport_id")
        .sort_values("molport_id"))

print(f"{len(hits)} compounds, each number and each molecule once: "
      f"{hits.molport_id.is_unique and hits.smiles.is_unique}")

## 5. Merge the same molecule written differently

Some entries are still the same substance, written in ways that do not match as text:

- **Salts:** a compound sold with a partner ion, such as a hydrochloride, carries an extra piece.
- **Charges:** the same group can be drawn with or without an extra hydrogen and a charge.
- **Tautomers:** some molecules shift a hydrogen from one atom to another. Both drawings are the
  same substance, swapping back and forth in water.

RDKit can rewrite a molecule into a single standard form that all three variants share. We use that
only to spot the duplicates; the list keeps the original text. Sorting by `rmsd` first means that
when two rows turn out to be the same substance, the better-fitting one survives.

> **Note:** this is the slowest cell in the notebook, about five minutes.

In [ ]:
from rdkit.Chem.MolStandardize import rdMolStandardize

uncharger, tautomers = rdMolStandardize.Uncharger(), rdMolStandardize.TautomerEnumerator()
tautomers.SetMaxTautomers(50)

def standardise(smiles):
    """Return one SMILES shared by all salt, charge and tautomer forms of a molecule."""
    mol = uncharger.uncharge(rdMolStandardize.FragmentParent(Chem.MolFromSmiles(smiles)))
    return Chem.MolToSmiles(tautomers.Canonicalize(mol))

ranked = hits.sort_values(["rmsd", "molport_id"]).reset_index(drop=True)
ranked["standard"] = ranked.smiles.map(standardise)
unique = ranked.drop_duplicates("standard").reset_index(drop=True)
print(f"{len(unique)} distinct substances ({len(ranked) - len(unique)} merged)")

## 6. Drop the near-copies

Many hits are small variations on one theme: a methyl group swapped for a chlorine, a ring atom
moved along one place. Ordering all of them teaches you little more than ordering one, so we keep
one compound per family.

**How similarity is measured.** A **fingerprint** records which small fragments a molecule contains,
as a row of 2,048 on/off switches. The **Tanimoto similarity** between two fingerprints is the share
of switches they have on in common: 1 means the same fragments, 0 means nothing shared. Above about
0.7, two molecules are usually variations on one theme.

**How one per family is kept.** RDKit's leader picker goes down the list in order. A compound is
kept unless it is 0.7 or more similar to one already kept. Because the list is sorted by `rmsd`,
each family is represented by its best-fitting member.

In [ ]:
from rdkit.Chem import rdFingerprintGenerator

morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fingerprints = [morgan.GetFingerprint(Chem.MolFromSmiles(s)) for s in unique.smiles]
print(f"{len(fingerprints)} fingerprints")

Now pick the leaders. The picker asks for a distance, which is 1 minus the similarity. This
takes about a minute and a half.

> **Exercise:** change `SIMILARITY` and run this cell again. A higher number is more permissive:
> on this data 0.8 keeps about 39,000 compounds, 0.6 about 19,000 and 0.5 about 12,000. Which
> would you send to a supplier, and why?

In [ ]:
from rdkit.SimDivFilters import rdSimDivPickers

SIMILARITY = 0.7
picks = rdSimDivPickers.LeaderPicker().LazyBitVectorPick(
    fingerprints, len(fingerprints), 1 - SIMILARITY, numThreads=8)
leaders = unique.loc[sorted(picks)]

print(f"{len(leaders)} compounds kept at Tanimoto {SIMILARITY}, from {len(unique)}")
print(f"median rmsd: {unique.rmsd.median():.3f} before, {leaders.rmsd.median():.3f} after")

## 7. Save the list

Two files go to `outputs/`, which is not part of the repository: the full list with catalogue
numbers, and the same compounds with only the molecules, which is the format the Ersilia Model Hub
takes.

Upload both to **Projects/BlueTeam/Data** in Drive when they are done, so the group can use them and
they can be copied into `data/` for the next notebook.

In [ ]:
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

final = leaders[["smiles", "molport_id"]].sort_values("molport_id")
final.to_csv(OUT / "pharmit_hits_molport.csv", index=False)
final[["smiles"]].to_csv(OUT / "pharmit_hits_smiles.csv", index=False)

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(OUT / "pharmit_hits_molport.csv"))
    files.download(str(OUT / "pharmit_hits_smiles.csv"))
final.head()

## Summary

- Read every pose Pharmit returned and collapsed the repeated shapes and mirror-image forms into
  one row per catalogue compound.
- Merged salts, charges and tautomers, which are the same substance written differently.
- Dropped near-copies at Tanimoto 0.7, keeping each family's best match, and saved
  `pharmit_hits_molport.csv` with the compounds and their catalogue numbers.

**Next:** open `blue_chemical_space.ipynb` to see where these compounds sit among known molecules,
and how they compare with silymarin.